# 01 · Reproduce everything from the raw clips

This notebook runs the whole pipeline on the 22 iPhone clips and ends by comparing your numbers with the
reference numbers in `results/`, which are the ones the README reports. Every step calls the same
command-line tool you can run by hand (`python -m press2sim ...`).

| Step | What it does | Time on 2 cores |
|---|---|---|
| 1 · Clips | checks that every clip in `trials.csv` is in `raw/` | seconds |
| 2 · Frame timing | reads each frame's own timestamp (iPhone slow motion drops frames) | 1 min |
| 3 · Calibration | checkerboard → pixels to millimetres, plus the face-plane height correction | a few minutes |
| 4 · Tracking | 105 dots and the bottle's two dots in every frame of 20 clips | about 45 min |
| 5 · Report | noise floor, repeat presses, one summary row per clip | seconds |
| 6 · Model | fits shape, damping and stiffness; predicts every press | about 20 min (a few minutes with `QUICK`) |
| 7 · Videos | the simulated block next to the real clip | a few minutes |
| 8 · MuJoCo | optional baseline and its follow-up checks | about 1.5 h |
| 8b · Model checks | optional: what the fitted model rests on (the README's limitations) | about 15 min |
| 9 · Compare | your numbers against `results/` | seconds |

More cores make tracking faster (`JOBS` below).

In [ ]:
from pathlib import Path
import csv, json, os, shutil, subprocess, sys, time

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)                                  # every command runs from the repo root
sys.path.insert(0, str(ROOT))

RAW = Path(os.environ.get("PRESS2SIM_RAW", "raw"))  # the original .MOV clips
OUT = Path("out")                               # everything this notebook writes
TRIALS = Path("trials.csv")                     # which clip is which
JOBS = min(4, os.cpu_count() or 1)               # clips tracked in parallel
QUICK = False      # True: reuse the fitted shape from results/ and refit only damping and stiffness
RUN_MUJOCO = False # True: also run the MuJoCo baseline and its follow-up checks (about 1.5 h more)
RUN_CHECKS = False # True: also run scripts/model_checks.py, the checks behind the README's limitations (about 15 min more)

In [ ]:
import importlib
for name, note in [("numpy", ""), ("scipy", ""), ("cv2", "OpenCV"), ("av", "PyAV, reads frame timestamps"),
                   ("matplotlib", ""), ("pyamg", "optional: about 10x faster model"),
                   ("mujoco", "optional: only for step 8")]:
    try:
        m = importlib.import_module(name)
        print(f"{name:11s} {getattr(m, '__version__', 'ok'):12s} {note}")
    except ImportError:
        print(f"{name:11s} {'missing':12s} {note}")
print(f"{'ffmpeg':11s} {'found' if shutil.which('ffmpeg') else 'not found':12s} optional: nicer video encoding")
print(f"python {sys.version.split()[0]}, {os.cpu_count()} cores, tracking {JOBS} clips at a time")

In [ ]:
def run(*args, script=None):
    """Run `python -m press2sim <args>` (or a script) from the repo root, streaming its output."""
    cmd = [sys.executable] + ([script] if script else ["-m", "press2sim"]) + [str(a) for a in args]
    print("$ python", " ".join(cmd[1:]), flush=True)
    t0 = time.time()
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
        for line in p.stdout:
            print(line, end="", flush=True)
    if p.returncode:
        raise RuntimeError(f"command failed (exit code {p.returncode})")
    print(f"--- done in {time.time() - t0:.0f} s")


def show(path, width=None):
    """Show a figure the pipeline wrote."""
    from IPython.display import Image, display
    display(Image(filename=str(path), width=width))

## 1 · The clips

Download the 22 clips (link in the README) into `raw/` next to this repo's `trials.csv`, or set the
environment variable `PRESS2SIM_RAW` to the folder they are in. Use the original files: re-encoding them
changes the frame timestamps.

In [ ]:
trials = list(csv.DictReader(open(TRIALS)))
found = [r for r in trials if (RAW / r["file"]).exists()]
missing = [r["file"] for r in trials if r not in found]
size = sum((RAW / r["file"]).stat().st_size for r in found)
print(f"{len(found)}/{len(trials)} clips found in {RAW} ({size / 1e9:.2f} GB)")
kinds = {}
for r in trials:
    kinds.setdefault(r["kind"], []).append(r["code"])
for k, v in kinds.items():
    print(f"  {k:9s} {', '.join(v)}")
assert not missing, f"missing clips: {missing}. Download them first (see the README)"


## 2 · Frame timing

iPhone "240 fps" slow motion drops frames, so the pipeline uses each frame's own timestamp. Expect an
average around 195–240 fps and some long gaps; both are normal.

In [ ]:
run("times", RAW, "--out", OUT)

## 3 · Calibration

The checkerboard clip C1 gives a homography from pixels to millimetres on the card. Rulers on the foam
then correct heights on the dotted face, which read about 5% short because the card was not exactly in
the face's plane. In this session the card's hold-out error is about 0.32 mm, above the 0.2 mm the tool
warns at, and C1 against C2 differ by about 0.4 mm; the README lists this under limitations.

In [ ]:
run("calib", "--trials", TRIALS, "--raw", RAW, "--out", OUT)
cal = json.loads((OUT / "calib" / "C1.json").read_text())
face = cal["face"]
print(f"hold-out error {cal['holdout_rms_mm']:.3f} mm, squares {cal['square_mm_measured_x']:.2f} x "
      f"{cal['square_mm_measured_z']:.2f} mm, {cal['px_per_mm']:.2f} px/mm")
print(f"face-plane height correction x{face['a_z']:.4f} (the two references agree within {face['spread_pct']:.1f}%)")
show(OUT / "qa" / "calib_C1.png", width=900)

## 4 · Tracking

One line per clip. Check: the rest frame has 105/105 dots, tracking is near 100%, rest noise is a few
hundredths of a millimetre, the camera stayed still, and the press clips report a sensible depth.

In [ ]:
run("track", "--trials", TRIALS, "--raw", RAW, "--out", OUT, "--jobs", JOBS)

In [ ]:
show(OUT / "qa" / "H1_frames.png", width=900)   # one held-out press: tracked dots at a few moments

## 5 · Report

The noise floor comes from S1, where nothing touches the foam. The repeat presses (R1 against T05, R2
against T08) went to different depths by hand, so their difference measures how different two hand
presses are, not tracking error.

In [ ]:
run("report", "--trials", TRIALS, "--out", OUT)

In [ ]:
import pandas as pd
cols = ["code", "kind", "rest_noise_mm", "dots_tracked_frac", "camera_max_shift_px", "max_depth_mm",
        "press_speed_mm_s", "hold_s"]
pd.read_csv(OUT / "summary.csv")[cols].round(3)

## 6 · Model

Three fitting steps, each on the observations that can set its parameters:

1. **Shape** (yield strain, plateau slope, densification, Poisson's ratio, the soft top layer, the cap's
   offset) from 16 pressing frames per training press, with the support from the dead-weight clips.
2. **Damping** (the relaxing fraction and its time) from the dent under the cap after the foam lets go of
   it. The dashpot is not fitted: these clips cannot tell 0 from 20 ms.
3. **Stiffness E** from the dead weights K1–K4, the only clips with a known force.

H1, O1 and F1 are never used for fitting. With `QUICK = True` the shape from `results/` is reused and
only steps 2 and 3 run.

In [ ]:
args = ["model", "--trials", TRIALS, "--raw", RAW, "--out", OUT]
if QUICK:
    (OUT / "model").mkdir(parents=True, exist_ok=True)
    shutil.copy(Path("results") / "model" / "start.json", OUT / "model" / "start.json")
    args.append("--keep-shape")
run(*args)

In [ ]:
fit = json.loads((OUT / "model" / "params.json").read_text())
p = fit["params"]
print("trained on:", ", ".join(fit["train"]), "| weights:", ", ".join(fit["weights"]),
      "| held out:", ", ".join(fit["held_out"]))
pd.DataFrame([
    ["Young's modulus E", f"{p['E'] / 1e3:.0f} kPa", "dead weights"],
    ["cells buckle at strain", f"{p['eps_y']:.3f}", "presses"],
    ["plateau slope", f"{p['h']:.3f} x E", "presses"],
    ["crushed cells touch at strain", f"{p['eps_d']:.2f}", "presses"],
    ["Poisson's ratio", f"{p['nu']:.2f}", "presses"],
    ["soft top layer", f"{p['skin_t']:.1f} mm at {p['skin_k']:.3f} x E", "presses"],
    ["support", f"{p['support']:.4f} x E per mm", "dead weights"],
    ["relaxing fraction, time", f"{p['g'][0]:.2f}, {p['tau'][0]:.2f} s", "the dent after release"],
    ["dashpot", f"{p['tau_v'] * 1e3:.2f} ms", "not fitted"],
], columns=["parameter", "value", "fitted on"])

In [ ]:
pred = pd.read_csv(OUT / "model" / "predictions.csv")
pred["error / movement"] = (100 * pred["relative_error"]).round(0).astype(int).astype(str) + "%"
pred[["clip", "set", "max_depth_mm", "rms_error_mm", "rms_movement_mm", "error / movement",
      "error_at_deepest_mm", "rms_error_no_damping_mm"]].round(2)

In [ ]:
show(OUT / "model" / "H1_timeseries.png", width=900)   # held-out press: measured against predicted

## 7 · Videos

`out/model/H1_simulation.mp4` and `O1_simulation.mp4` show the real clip with tracked and predicted dots
next to the simulated 3D block, both driven only by the cap's measured depth. One frame:

In [ ]:
run("simulate", "--trials", TRIALS, "--raw", RAW, "--out", OUT)
import cv2
import matplotlib.pyplot as plt
cap = cv2.VideoCapture(str(OUT / "model" / "H1_simulation.mp4"))
cap.set(cv2.CAP_PROP_POS_MSEC, 4500)
ok, frame = cap.read()
plt.figure(figsize=(12, 6.75)); plt.imshow(frame[..., ::-1]); plt.axis("off"); plt.show()

## 8 · MuJoCo baseline (optional)

The same block in MuJoCo 4.0, scored the same way, then the follow-up checks (finer grid, MuJoCo's
plasticity, its Stable Neo-Hookean material). Set `RUN_MUJOCO = True` above to run it.

In [ ]:
if RUN_MUJOCO:
    run("mujoco", "--trials", TRIALS, "--raw", RAW, "--out", OUT)
    run("--trials", TRIALS, "--out", OUT, script="scripts/mujoco_checks.py")
else:
    print("skipped: set RUN_MUJOCO = True to run it (the reference results are in results/model/)")

## 8b · Model checks (optional)

`scripts/model_checks.py` reruns the checks behind the README's limitations on your own fit: the soft
top layer under the cap, how much of the cap's travel reaches the dots, the stiffness under other
assumptions about the block, and what the dent after release says about damping. Set
`RUN_CHECKS = True` above to run it. `--part numerics` (finer grids, about 40 min) is separate.

In [ ]:
if RUN_CHECKS:
    run("--trials", TRIALS, "--out", OUT, script="scripts/model_checks.py")
else:
    print("skipped: set RUN_CHECKS = True to run it (the reference results are in results/model/model_checks.json)")

## 9 · Your numbers against the reference

Tracking and fitting are deterministic, so the same clips should give the same numbers up to small
floating-point differences between machines and library versions.

In [ ]:
ref = pd.read_csv(Path("results") / "model" / "predictions.csv").set_index("clip")
new = pd.read_csv(OUT / "model" / "predictions.csv").set_index("clip")
cmp = pd.DataFrame({"set": new["set"], "your RMS error (mm)": new["rms_error_mm"],
                    "reference (mm)": ref["rms_error_mm"].reindex(new.index)})
cmp["difference (mm)"] = cmp["your RMS error (mm)"] - cmp["reference (mm)"]
display(cmp.round(3))
ref_p = json.loads((Path("results") / "model" / "params.json").read_text())["params"]
for k in ["E", "eps_y", "h", "eps_d", "nu", "g", "tau"]:
    mine, theirs = (p[k][0], ref_p[k][0]) if isinstance(p[k], list) else (p[k], ref_p[k])
    print(f"{k:6s} yours {mine:.6g}   reference {theirs:.6g}")
worst = cmp["difference (mm)"].abs().max()
print(f"\nlargest difference in RMS error: {worst:.3f} mm ->",
      "reproduced" if worst < 0.05 else "check the differences above")